In [ ]:
# Install necessary libraries
!pip install --upgrade transformers datasets scikit-learn accelerate

import warnings
warnings.filterwarnings("ignore")
print("Setup complete. Checking GPU...")
!nvidia-smi

In [ ]:
# 1. Fix Protobuf version (This is the root cause of the known error)
!pip install protobuf==3.20.3

# 2. Suppress WandB and other logging tools (To reduce noise)
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3" # Hide TensorFlow warnings

In [ ]:
import torch
import sys

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")

# GPU Check
if torch.cuda.is_available():
    print(f"✅ GPU Active: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU Not Found! (Did you select T4 x2 in Accelerator settings?)")

# Confirming PyTorch usage regardless of TensorFlow installation
try:
    import tensorflow as tf
    print(f"Note: TensorFlow {tf.__version__} is installed in the environment, but we will not be using it.")
except ImportError:
    print("TensorFlow is not installed (No issue).")

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import json
import gc
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

# --- SETTINGS ---
MODEL_NAME = "microsoft/unixcoder-base"
MAX_LENGTH = 256
BATCH_SIZE = 32
EPOCHS = 1         # 2 epochs is enough for a quick run
LEARNING_RATE = 2e-5

# Data Paths (Check Kaggle path)
INPUT_DIR = '/kaggle/input/sem-eval-2026-task-13-subtask-b/Task_B/'
TRAIN_PATH = os.path.join(INPUT_DIR, 'train.parquet')
VAL_PATH = os.path.join(INPUT_DIR, 'validation.parquet')
TEST_PATH = os.path.join(INPUT_DIR, 'test.parquet')

id2label = {
    0: "human", 1: "deepseek", 2: "qwen", 3: "01-ai", 4: "bigcode",
    5: "gemma", 6: "phi", 7: "meta-llama", 8: "ibm-granite", 9: "mistral", 10: "openai"
}
label2id = {v: k for k, v in id2label.items()}

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Calculate Cross-Entropy Loss
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        # Calculate focusing parameter
        pt = torch.exp(-ce_loss)
        # Calculate Focal Loss
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

class ImbalancedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights is not None:
            # Move class weights to the appropriate device
            self.class_weights = torch.tensor(class_weights, dtype=torch.float32).to(self.args.device)
        else:
            self.class_weights = None

        # Initialize Focal Loss with class weights
        self.focal_loss = FocalLoss(alpha=self.class_weights, gamma=2.0)

    # Override compute_loss to use Focal Loss
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # Compute loss using the custom FocalLoss
        loss = self.focal_loss(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
def load_and_prepare_data():
    print(f"Loading data: {TRAIN_PATH}")
    df = pd.read_parquet(TRAIN_PATH)
    val_df = pd.read_parquet(VAL_PATH) # Original has 100k data

    # Cleaning
    df = df.dropna(subset=['code', 'label'])
    val_df = val_df.dropna(subset=['code', 'label'])
    df['label'] = df['label'].astype(int)
    val_df['label'] = val_df['label'].astype(int)

    # --- 1. TRAIN PRUNING (Already done) ---
    df_ai = df[df['label'] > 0]
    # Sample 100,000 human examples for balance
    df_human = df[df['label'] == 0].sample(n=100000, random_state=42)
    # Concatenate and shuffle the training set
    train_df = pd.concat([df_ai, df_human]).sample(frac=1, random_state=42).reset_index(drop=True)

    # --- 2. VALIDATION PRUNING (NEW ADDITION) ✂️ ---
    # Sample only 10,000 examples instead of 100,000.
    # This is sufficient to observe the trend.
    val_df = val_df.sample(n=10000, random_state=42).reset_index(drop=True)
    # --------------------------------------------

    print(f"⚠️ OPTIMIZED:")
    print(f"   - Train Size: {len(train_df)}")
    print(f"   - Val Size: {len(val_df)} (Reduced from 100k to 10k)")

    # Compute Class Weights
    labels = train_df['label'].values
    classes = np.unique(labels)
    # Calculate balanced weights for Focal Loss
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=labels)

    return train_df, val_df, weights

# Rerun to load data with new validation size
train_df, val_df, class_weights = load_and_prepare_data()

# Don't forget to recreate Datasets and Tokenizer!
# (This part needs to be run again in Cell 7)

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    return tokenizer(
        examples["code"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

print("Preparing Dataset...")
train_dataset = Dataset.from_pandas(train_df[['code', 'label']])
val_dataset = Dataset.from_pandas(val_df[['code', 'label']])

# Fast tokenization with num_proc=4
tokenized_train = train_dataset.map(preprocess_function, batched=True, num_proc=4)
tokenized_val = val_dataset.map(preprocess_function, batched=True, num_proc=4)

tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_val = tokenized_val.rename_column("label", "labels")
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_val.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=11,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    # Use macro average for F1-score to handle class imbalance
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    return {'accuracy': acc, 'f1_macro': f1}

training_args = TrainingArguments(
    output_dir="./unixcoder_focal_v0",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=800,
    save_strategy="steps",
    save_steps=800,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,
    fp16=True, # Enable mixed precision training
    report_to="none"
)

# Initialize the custom trainer with class weights
trainer = ImbalancedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("🚀 Training Starts...")
trainer.train()
# Save the final best model and tokenizer
trainer.save_model("./final_best_model")
tokenizer.save_pretrained("./final_best_model")
print("✅ Training Complete.")

In [ ]:
print("Processing test data...")
test_df = pd.read_parquet(TEST_PATH)
test_dataset = Dataset.from_pandas(test_df)
# Tokenize the test data
tokenized_test = test_dataset.map(preprocess_function, batched=True, num_proc=4)

print("Generating predictions...")
# Get predictions from the trained model
predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=1)

# Create submission file
submission = pd.DataFrame()
if 'ID' in test_df.columns:
    submission['ID'] = test_df['ID']
elif 'id' in test_df.columns:
    submission['ID'] = test_df['id']
else:
    submission['ID'] = test_df.index
submission['label'] = preds
submission.to_csv('submission.csv', index=False)
print("✅ submission.csv is ready.")

In [ ]:
# Memory cleanup
import gc
del tokenized_train, train_df
gc.collect()
torch.cuda.empty_cache()

def mine_hard_examples_full_data(trainer):
    print("⛏️ Starting hard example mining on the original 500k data...")

    # 1. Read the full data (Unpruned)
    full_df = pd.read_parquet(TRAIN_PATH)
    # Clean NaNs and reset index (to maintain original indexing)
    full_df = full_df.dropna(subset=['code', 'label']).reset_index(drop=True)

    print(f"Total Data to Scan: {len(full_df)}")

    # 2. Prepare Dataset
    full_dataset = Dataset.from_pandas(full_df[['code', 'label']])
    tokenized_full = full_dataset.map(preprocess_function, batched=True, num_proc=4)

    # 3. Get Predictions
    print("Generating predictions (This may take a while)...")
    predictions = trainer.predict(tokenized_full)

    # Convert to PyTorch tensors for loss calculation
    logits = torch.tensor(predictions.predictions)
    labels = torch.tensor(predictions.label_ids)

    # Save Logits (for Ensemble)
    np.save("v0_full_logits.npy", predictions.predictions)
    print("💾 Logits saved.")

    # 4. Calculate Loss (CrossEntropyLoss)
    loss_fct = nn.CrossEntropyLoss(reduction='none')
    losses = loss_fct(logits, labels)

    # 5. Sort by loss
    sorted_losses, sorted_indices = torch.sort(losses, descending=True)

    # Take the top 20%
    hard_count = int(len(full_df) * 0.20)
    hard_indices = sorted_indices[:hard_count].tolist()
    easy_indices = sorted_indices[hard_count:].tolist()

    # 6. Save as JSON
    data_to_save = {"hard_indices": hard_indices, "easy_indices": easy_indices}
    with open("hard_negative_indices.json", 'w') as f:
        json.dump(data_to_save, f)

    print(f"✅ MINING COMPLETE!")
    print(f"Hard Set: {len(hard_indices)} | Max Loss: {sorted_losses[0]:.4f}")

mine_hard_examples_full_data(trainer)

In [ ]:
# --- BLOCK 12: DETAILED PERFORMANCE REPORT (QUICK) ---
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def print_validation_report(trainer, val_dataset):
    print("🔍 Measuring performance on the validation set (Quick)...")

    # 1. Get Predictions (Validation Set Only - Fast)
    predictions = trainer.predict(val_dataset)
    preds = np.argmax(predictions.predictions, axis=1)
    labels = predictions.label_ids

    # 2. Get Class Names
    target_names = [id2label[i] for i in range(len(id2label))]

    # 3. Text Report (Classification Report)
    print("\n" + "="*60)
    print("🎓 MODEL REPORT CARD (VALIDATION SET)")
    print("="*60)

    # digits=4 for precise decimal display
    report = classification_report(labels, preds, target_names=target_names, digits=4)
    print(report)
    print("="*60 + "\n")

    # 4. Where Did We Fail? (Critical Analysis)
    print("📉 CRITICAL ANALYSIS (Challenging Classes):")

    # Get the report as a dictionary for analysis
    report_dict = classification_report(labels, preds, target_names=target_names, output_dict=True)

    for class_name in target_names:
        if class_name == "human": continue # Human class is usually good

        f1 = report_dict[class_name]['f1-score']
        recall = report_dict[class_name]['recall']

        # Check for weak performance (F1 < 0.70)
        if f1 < 0.70:
            print(f"⚠️  ATTENTION: '{class_name}' class is weak! (F1: {f1:.3f}, Recall: {recall:.3f})")
            print(f"    -> Reason: The model might be mistaking this class for 'Human'.")
        else:
            print(f"✅  SUCCESS: '{class_name}' is well-learned. (F1: {f1:.3f})")

# Run the report
print_validation_report(trainer, tokenized_val)